[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berlin1906/genesis_env/blob/main/training/self_improve.ipynb)

# Genesis Self-Improvement Loop

Interactive notebook for running and monitoring the self-improvement cycle.

**Flow**: evaluate baseline → Tool Architect rewrites/creates tools → evaluate again → revert if worse

| Cell | What it does |
|------|-------------|
| 1 | Setup & config |
| 2 | Run a single inference episode & inspect tool log |
| 3 | Baseline evaluation (N episodes) |
| 4 | Tool Architect — dry-run decision |
| 5 | Full improvement cycle (eval → architect → eval) |
| 6 | Run N cycles end-to-end |
| 7 | Loop history & reward plots |

## Cell 1 — Setup

In [ ]:
!git clone https://huggingface.co/spaces/berlin1906/genesis_env

In [ ]:
import os
%cd genesis_env

In [ ]:
!pip install -U -r requirements.txt

In [ ]:
%env HF_TOKEN=<YOUR_HF_TOKEN>
%env API_BASE_URL=https://router.huggingface.co/v1
%env MODEL_NAME=Qwen/Qwen3.5-4B

In [ ]:
import sys, os, json
from pathlib import Path

# Make sure the project root is on the path
PROJECT_ROOT = Path(".").resolve().parent  # notebook lives in training/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# ── Config ────────────────────────────────────────────────────────────────
N_EPISODES   = 3    # episodes per evaluation phase
N_CYCLES     = 3    # number of improve→eval cycles
DRY_RUN      = True # set False to actually write tool files
SEED_OFFSET  = 0    # change to run on different benchmark tasks

print(f"Project root : {PROJECT_ROOT}")
print(f"API base     : {os.getenv('API_BASE_URL')}")
print(f"Model        : {os.getenv('MODEL_NAME')}")
print(f"n_episodes={N_EPISODES}  n_cycles={N_CYCLES}  dry_run={DRY_RUN}")

## Cell 2 — Single Inference Episode

Run one episode to inspect the tool call log and per-step rewards.

In [ ]:
from openai import OpenAI
from agent.tool_executor import ToolExecutor
from envs.gen_env.models import GenEnvAction
from envs.gen_env.server.gen_env_environment import GenesisEnvironment
from inference import run_tool_loop

client   = OpenAI(base_url=os.getenv("API_BASE_URL"), api_key=os.getenv("HF_TOKEN") or os.getenv("API_KEY", ""))
executor = ToolExecutor()
env      = GenesisEnvironment()

executor.reset_log()
obs = env.reset(seed=SEED_OFFSET)

print(f"Task : {obs.task_id}")
print(f"Diff : {obs.difficulty}")
print(f"Desc : {obs.task_description[:200]}")
print()

final_code, step_log = run_tool_loop(client, executor, obs.task_description, obs.starter_code, env=env)
tool_log = executor.get_log()

action   = GenEnvAction(code=final_code, task_id=obs.task_id, tool_usage_log=tool_log)
step_obs = env.step(action)

print("\n── Episode result ──────────────────────────")
print(f"reward       : {step_obs.reward:.4f}")
print(f"tests passed : {step_obs.tests_passed}/{step_obs.tests_total}")
print(f"nl_feedback  : {step_obs.nl_feedback[:200]}")
print(f"tool_grades  : {step_obs.tool_grades}")
print(f"tool_weights : {step_obs.tool_weights}")

In [ ]:
# Inspect the per-call tool log
print(f"{'#':<3} {'Tool':<25} {'Grade':>6}  Result preview")
print("-" * 80)
for i, entry in enumerate(tool_log):
    grade = entry.get("grade", "?")
    grade_str = f"{grade:.2f}" if isinstance(grade, float) else str(grade)
    result_preview = str(entry.get("result", ""))[:60].replace("\n", " ")
    print(f"{i:<3} {entry['tool']:<25} {grade_str:>6}  {result_preview}")

## Cell 3 — Baseline Evaluation

Run `N_EPISODES` episodes to establish baseline reward and tool weights.

In [ ]:
from training.self_improve import evaluate

seeds_baseline = list(range(SEED_OFFSET, SEED_OFFSET + N_EPISODES))
baseline = evaluate(N_EPISODES, seeds=seeds_baseline)

print(f"\nBaseline mean reward : {baseline['mean_reward']:.4f}")
print(f"Episodes completed   : {baseline['n_episodes']}")
print("\nTool weights:")
for tool, weight in sorted(baseline["tool_weights"].items()):
    bar = "█" * int(weight * 20) + "░" * (20 - int(weight * 20))
    print(f"  {tool:<25} {weight:.3f}  {bar}")

## Cell 4 — Tool Architect (dry-run)

See what the Architect would do without writing any files.

In [ ]:
from training.tool_architect import apply_improvement, decide_action
from envs.gen_env.server.tool_registry import ToolRegistry, ToolFlag

registry = ToolRegistry()
registry.ema_weights.update(baseline["tool_weights"])
tool_flags = {t: registry.flag(t).value for t in baseline["tool_weights"]}

print("Tool flags:")
for t, f in tool_flags.items():
    icon = {"KEEP": "✓", "REVIEW": "~", "REPLACE": "✗"}.get(f, "?")
    print(f"  {icon} {t:<25}  {f}")

# Dry-run: see decision without writing
improvement = apply_improvement(
    tool_weights=baseline["tool_weights"],
    tool_flags=tool_flags,
    nl_feedback=baseline["nl_feedback"],
    recent_tool_calls=baseline["tool_logs"],
    dry_run=True,
)

print("\nArchitect decision:")
print(json.dumps(improvement, indent=2))

## Cell 5 — One Improvement Cycle

Runs: baseline eval → Architect (respects `DRY_RUN`) → post-improvement eval → revert if worse.

In [ ]:
from training.self_improve import evaluate, _revert_tool
from training.tool_architect import apply_improvement
from envs.gen_env.server.tool_registry import ToolRegistry

CYCLE = 1  # label for this manual cycle

# ── Before ──
before_seeds = list(range(CYCLE * 100, CYCLE * 100 + N_EPISODES))
before = evaluate(N_EPISODES, seeds=before_seeds)
print(f"[BEFORE] mean_reward={before['mean_reward']:.4f}")

# ── Architect ──
registry = ToolRegistry()
registry.ema_weights.update(before["tool_weights"])
tool_flags = {t: registry.flag(t).value for t in before["tool_weights"]}

improvement = apply_improvement(
    tool_weights=before["tool_weights"],
    tool_flags=tool_flags,
    nl_feedback=before["nl_feedback"],
    recent_tool_calls=before["tool_logs"],
    dry_run=DRY_RUN,
)
print(f"[ARCHITECT] {improvement['action']} → {improvement.get('target_tool','?')}  success={improvement['success']}")

# ── After ──
after_seeds = list(range(CYCLE * 100 + 50, CYCLE * 100 + 50 + N_EPISODES))
after = evaluate(N_EPISODES, seeds=after_seeds)
delta = after["mean_reward"] - before["mean_reward"]
print(f"[AFTER]  mean_reward={after['mean_reward']:.4f}  delta={delta:+.4f}")

# ── Revert if worse ──
reverted = False
if delta < 0 and not DRY_RUN:
    file_written = improvement.get("file_written")
    if file_written:
        reverted = _revert_tool(file_written)

status = "REVERTED" if reverted else ("improved" if delta >= 0 else "no backup to revert")
print(f"\nCycle {CYCLE} complete — delta={delta:+.4f}  status={status}")

## Cell 6 — Full N-Cycle Loop

Runs the complete self-improvement loop for `N_CYCLES` cycles. State is persisted to `training/loop_state.json`.

In [ ]:
from training.self_improve import run_loop

run_loop(n_episodes=N_EPISODES, n_cycles=N_CYCLES, dry_run=DRY_RUN)

## Cell 7 — Loop History & Reward Plots

In [ ]:
import json
from pathlib import Path

state_file = Path("loop_state.json")  # relative to training/
if not state_file.exists():
    state_file = PROJECT_ROOT / "genesis_env" / "training" / "loop_state.json"
state = json.loads(state_file.read_text()) if state_file.exists() else {"cycle": 0, "history": []}
history = state["history"]

print(f"Total cycles recorded: {len(history)}")
print()
print(f"{'Cycle':>6} {'Before':>8} {'After':>8} {'Delta':>8} {'Rev':>5}  Action")
print("-" * 60)
for h in history:
    rev = "yes" if h.get("reverted") else "no"
    print(f"{h['cycle']:6d} {h['before_reward']:8.4f} {h['after_reward']:8.4f} {h['delta']:+8.4f} {rev:>5}  {h['architect_action']}:{h.get('target_tool','?')}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    if not history:
        print("No history yet — run Cell 6 first.")
    else:
        cycles  = [h["cycle"] for h in history]
        befores = [h["before_reward"] for h in history]
        afters  = [h["after_reward"] for h in history]
        deltas  = [h["delta"] for h in history]
        colors  = ["red" if h.get("reverted") else ("green" if h["delta"] >= 0 else "orange") for h in history]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

        ax1.plot(cycles, befores, "o--", label="Before", color="steelblue", linewidth=1.5)
        ax1.plot(cycles, afters,  "o-",  label="After",  color="darkorange", linewidth=1.5)
        ax1.set_ylabel("Mean reward")
        ax1.set_title("Self-Improvement Loop — Reward per Cycle")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.bar(cycles, deltas, color=colors)
        ax2.axhline(0, color="black", linewidth=0.8)
        ax2.set_ylabel("Delta (after − before)")
        ax2.set_xlabel("Cycle")
        ax2.grid(True, alpha=0.3, axis="y")
        patches = [
            mpatches.Patch(color="green",  label="improved"),
            mpatches.Patch(color="orange", label="worse (kept)"),
            mpatches.Patch(color="red",    label="worse (reverted)"),
        ]
        ax2.legend(handles=patches, fontsize=9)

        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / "genesis_env" / "training" / "loop_history.png", dpi=120)
        plt.show()
        print("Plot saved to genesis_env/training/loop_history.png")
except ImportError:
    print("matplotlib not installed — pip install matplotlib to enable plots")

## Cell 8 — Tool Weight History

Peek at current EMA weights directly from the tool registry.

In [ ]:
from envs.gen_env.server.tool_registry import ToolRegistry, ToolFlag
from agent.tool_executor import _discover_tools

registry = ToolRegistry()
snapshot = registry.snapshot()

print(f"{'Tool':<25} {'Weight':>7}  {'Flag':<8}")
print("-" * 45)
for tool in _discover_tools():
    w = snapshot.get(tool, 0.5)
    flag = registry.flag(tool).value
    bar = "█" * int(w * 20) + "░" * (20 - int(w * 20))
    print(f"{tool:<25} {w:7.3f}  {flag:<8}  {bar}")

---
## GRPO Fine-Tuning

The cells below fine-tune the LLM that drives the agent using **Group Relative Policy Optimization (GRPO)**.
For each task we sample `GROUP_SIZE` completions, score each via the Genesis environment reward,
then update the model using group-normalised advantages — no separate critic needed.

> **Requirements** (GPU recommended):
> ```bash
> pip install transformers peft trl accelerate bitsandbytes
> ```
> On Mac / CPU-only machines set `LOAD_IN_4BIT = False` and `USE_LORA = False`.

## Cell 9 — GRPO Setup

In [ ]:
!pip install -U transformers peft trl accelerate itsandbytes>=0.46.1

In [ ]:
from training.grpo_trainer import GRPOTrainer, GRPOConfig

# ── GRPO config ────────────────────────────────────────────────────────────
GRPO_MODEL   = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-Coder-7B-Instruct")
GROUP_SIZE   = 4      # completions sampled per prompt (G)
LR           = 5e-6
KL_COEF      = 0.04
LOAD_IN_4BIT = True   # QLoRA — set False on CPU / Mac
USE_LORA     = True
BATCH_SIZE   = 4      # tasks per GRPO batch (per combined cycle)
GRPO_OUTPUT  = str(PROJECT_ROOT / "checkpoints" / "grpo")

grpo_cfg = GRPOConfig(
    model_name=GRPO_MODEL,
    group_size=GROUP_SIZE,
    lr=LR,
    kl_coef=KL_COEF,
    load_in_4bit=LOAD_IN_4BIT,
    use_lora=USE_LORA,
    output_dir=GRPO_OUTPUT,
)

print("Initialising GRPOTrainer...")
grpo_trainer = GRPOTrainer(grpo_cfg)
print("GRPOTrainer ready.")

## Cell 10 — Single GRPO Batch Step

Fine-tune on one batch of `BATCH_SIZE` tasks and inspect per-completion rewards.

In [ ]:
from training.combined_loop import _load_tasks, _pick_batch

all_tasks = _load_tasks()
batch     = _pick_batch(all_tasks, cycle_num=1, batch_size=BATCH_SIZE)

print(f"Batch tasks: {[t['id'] for t in batch]}\n")

if not DRY_RUN:
    grpo_result = grpo_trainer.train_batch(batch)
    print(f"\nGRPO batch result:")
    print(f"  mean_loss   : {grpo_result['mean_loss']:.4f}")
    print(f"  mean_reward : {grpo_result['mean_reward']:.4f}")
    print(f"  n_tasks     : {grpo_result['n_tasks']}")
else:
    print("(dry_run=True) Skipping GRPO update — set DRY_RUN=False in Cell 1 to train.")

## Cell 11 — Full Combined Loop

Runs `N_CYCLES` cycles of: **GRPO batch → self-improve (eval → architect → eval → revert)**.

State is persisted to `training/combined_state.json`.

In [ ]:
from training.combined_loop import run_combined_loop

run_combined_loop(
    n_episodes=N_EPISODES,
    n_cycles=N_CYCLES,
    batch_size=BATCH_SIZE,
    dry_run=DRY_RUN,
    grpo_only=False,
    improve_only=False,
    grpo_cfg=grpo_cfg,
)

## Cell 12 — Combined Metrics Plot

Loads `combined_state.json` and plots GRPO reward + self-improvement delta on a 3-panel chart.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    state_file = PROJECT_ROOT / "training" / "combined_state.json"
    if not state_file.exists():
        print("combined_state.json not found — run Cell 11 first.")
    else:
        history      = json.loads(state_file.read_text())["history"]
        cycles       = [h["cycle"] for h in history]
        grpo_rewards = [h.get("grpo_mean_reward") for h in history]
        befores      = [h.get("before_reward") for h in history]
        afters       = [h.get("after_reward") for h in history]
        deltas       = [h.get("delta", 0) or 0 for h in history]
        colors       = [
            "red" if h.get("reverted") else
            ("green" if (h.get("delta") or 0) >= 0 else "orange")
            for h in history
        ]

        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

        ax1.plot(cycles, grpo_rewards, "o-", color="steelblue", linewidth=1.5, label="GRPO mean reward")
        ax1.set_ylabel("GRPO reward")
        ax1.set_title("Combined GRPO + Self-Improvement Loop")
        ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

        ax2.plot(cycles, befores, "o--", color="grey",       linewidth=1.5, label="Before (eval)")
        ax2.plot(cycles, afters,  "o-",  color="darkorange", linewidth=1.5, label="After (eval)")
        ax2.set_ylabel("Self-improve reward")
        ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

        ax3.bar(cycles, deltas, color=colors)
        ax3.axhline(0, color="black", linewidth=0.8)
        ax3.set_ylabel("Delta"); ax3.set_xlabel("Cycle")
        ax3.grid(True, alpha=0.3, axis="y")
        ax3.legend(handles=[
            mpatches.Patch(color="green",  label="improved"),
            mpatches.Patch(color="orange", label="worse (kept)"),
            mpatches.Patch(color="red",    label="worse (reverted)"),
        ], fontsize=9)

        plt.tight_layout()
        out = PROJECT_ROOT / "training" / "combined_history.png"
        plt.savefig(out, dpi=120)
        plt.show()
        print(f"Plot saved to {out}")
except ImportError:
    print("matplotlib not installed — pip install matplotlib")